# Page 12: which chains become lessons?

STaR (Zelikman et al. 2022) trains a model on its own reasoning. The model answers each problem,
an answer key throws away the chains that reached the wrong answer, and the survivors become the
next training set. This notebook runs that filter without training anything. It reuses page 7's
pile, where Qwen2.5-3B-Instruct answered 100 hard MATH problems 16 times each, and asks which of
those chains STaR would keep.

Two rules decide what is kept:

| Rule | Keeps a chain when | Stands for |
|---|---|---|
| `key` | its answer matches the reference answer | STaR as published |
| `popular` | its answer is the one most of the problem's chains gave | STaR with the answer key taken away |

A third section runs STaR's rationalization step for real: every problem the model's greedy chain
got wrong is asked again with the right answer given as a hint.

**What it writes**, in `out/page_12/`:

| File | What it holds |
|---|---|
| `results.json` | every number this notebook computes |
| `manifest.json` | how, where and with what the run was made |
| `hint_traces.jsonl` | the hinted chains, when the hint retry has been run |
| `figure_harvest.png` | the figure below |

`page_12_site_data.py` turns these into the `derived.json` the website draws. Shared helpers live in
`harness.py`. Both ship beside this notebook.

## 1. How to run it

| Mode | What it does | Hardware | Time |
|---|---|---|---|
| `REPLAY` (default) | Runs the filter over page 7's chains, and reads the hint retry from `hint_traces.jsonl` if it is there. Nothing is generated. | any laptop | seconds |
| `QUICK` | Also generates the hint retry for 5 problems. | GPU, ~8 GB | a few minutes |
| `HINT` | Also generates the hint retry for all 100 problems. | GPU, ~8 GB | 20 to 40 minutes |

Page 7's chains are read from `out/page_07/traces.jsonl`. Run `page_07_frequency_voting.ipynb`
first, or download its bundle and unpack it beside this notebook.

```bash
PAGE12_RUN_MODE=HINT jupyter nbconvert --to notebook --execute page_12_star_harvest.ipynb
```

The hint retry is resumable: each chain is appended to `hint_traces.jsonl` as soon as it is written.

In [ ]:
import os

RUN_MODE = "REPLAY"                                       # "REPLAY", "QUICK" or "HINT"
RUN_MODE = os.environ.get("PAGE12_RUN_MODE", RUN_MODE)    # or choose it from a shell

POOL_CONFIG = "t0.30-c2048"      # page 7's pile: 16 chains per problem at temperature 0.3
GREEDY_CONFIG = "t0.00-c2048"    # page 7's greedy chain: one per problem
N_CHAINS = 16                    # the first 16 chains by sample index
HINT_CAP = 2048                  # the hinted chain gets the same cap as page 7's chains
QUICK_ITEMS = 5

assert RUN_MODE in ("REPLAY", "QUICK", "HINT"), RUN_MODE

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import harness
from reasoning_from_scratch.ch03 import extract_final_candidate, get_last_boxed, grade_answer, render_prompt

OUT_DIR = Path("out/page_12")
OUT_DIR.mkdir(parents=True, exist_ok=True)
P7_TRACES = Path("out/page_07/traces.jsonl")
HINT_PATH = OUT_DIR / "hint_traces.jsonl"
RESULTS_PATH = OUT_DIR / "results.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
FIGURE_PATH = OUT_DIR / "figure_harvest.png"

print("harness", harness.fingerprint()["version"], "| mode", RUN_MODE)

## 2. Page 7's pile

Page 7's log is append-only, so a key can appear twice; the last write wins. Only the run that
holds most of the records is read. Each problem keeps its first 16 chains by sample index, and its
greedy chain.

In [ ]:
assert P7_TRACES.exists(), f"page 7's chains are not at {P7_TRACES}; see section 1"

latest = {}
for line in P7_TRACES.open(encoding="utf-8"):
    if line.strip():
        r = json.loads(line)
        latest[(r["run_id"], r["config_id"], r["problem_id"], r["sample_index"])] = r
records = list(latest.values())
SOURCE_RUN_ID = Counter(r["run_id"] for r in records).most_common(1)[0][0]
records = [r for r in records if r["run_id"] == SOURCE_RUN_ID]

pool, greedy = {}, {}
for r in sorted(records, key=lambda r: r["sample_index"]):
    if r["config_id"] == POOL_CONFIG:
        pool.setdefault(r["problem_id"], []).append(r)
    elif r["config_id"] == GREEDY_CONFIG:
        greedy.setdefault(r["problem_id"], r)
ITEMS = sorted(pool)
pool = {i: pool[i][:N_CHAINS] for i in ITEMS}

assert all(len(pool[i]) == N_CHAINS for i in ITEMS), "a problem has fewer than 16 chains"
assert set(greedy) >= set(ITEMS), "a problem has no greedy chain"
MODEL_ID = pool[ITEMS[0]][0]["model_id"]

print(f"run      {SOURCE_RUN_ID}")
print(f"model    {MODEL_ID}")
print(f"{len(ITEMS)} problems x {N_CHAINS} chains at {POOL_CONFIG}, plus one greedy chain each")
print(f"chains right: {sum(r['correct'] for i in ITEMS for r in pool[i])} of {len(ITEMS) * N_CHAINS}")

## 3. Answer classes

The `popular` rule counts answers, so equivalent answers have to count as one. Two answers are in
the same class when chapter 3's grader accepts either as the other, which is page 7's rule. A chain
with no answer is in no class.

In [ ]:
_EQ = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ:
        try:
            _EQ[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ[k] = False
    return _EQ[k]


def classes(keys):
    """A class index per answer (-1 for no answer), and each class's first-seen label."""
    labels, order = [], []
    for k in keys:
        if k is None:
            labels.append(-1)
            continue
        hit = next((j for j, lab in enumerate(order) if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = len(order) - 1
        labels.append(hit)
    return labels, order


print(classes([r"\frac{1}{2}", "0.5", "3", None, "1,024", "1024"]))

rows = {}
for i in ITEMS:
    cls, labels = classes([r["vote_key"] for r in pool[i]])
    rows[i] = {"correct": [bool(r["correct"]) for r in pool[i]], "cls": cls, "labels": labels,
               "greedy_correct": bool(greedy[i]["correct"])}

# One class is either all right or all wrong.
for i, row in rows.items():
    for c in set(row["cls"]) - {-1}:
        assert len({ok for ok, cc in zip(row["correct"], row["cls"]) if cc == c}) == 1, i
print(f"{sum(len(r['labels']) for r in rows.values())} answer classes over {len(ITEMS)} problems")

## 4. The filter

`harvest` returns the chains among a problem's first `k` that a rule keeps. Under `popular`, a tie
is not a vote: when two answers share the top count, nothing is kept.

In [ ]:
def popular(cls, k):
    """The class most of the first k chains agree on; a tie, or no answers, is None."""
    tally = Counter(c for c in cls[:k] if c >= 0)
    if not tally:
        return None
    (top, n), *rest = tally.most_common()
    if rest and rest[0][1] == n:
        return None
    return top


def harvest(row, k, rule):
    """The chains among the first k that the filter keeps, as indices."""
    if rule == "key":
        return [j for j in range(k) if row["correct"][j]]
    top = popular(row["cls"], k)
    return [] if top is None else [j for j in range(k) if row["cls"][j] == top]


toy = {"correct": [False, True, False, True], "cls": [0, 1, 0, 1]}
assert harvest(toy, 4, "key") == [1, 3]
assert harvest(toy, 4, "popular") == []          # 2 votes each: a tie
assert harvest(toy, 3, "popular") == [0, 2]      # the wrong answer wins

## 5. What each rule keeps

For every number of attempts from 1 to 16: how many problems yield at least one lesson, how many
of those lessons are entirely wrong, and how many chains are kept in all.

In [ ]:
curves = {}
for rule in ("key", "popular"):
    pts = []
    for k in range(1, N_CHAINS + 1):
        kept = {i: harvest(rows[i], k, rule) for i in ITEMS}
        wrong = {i: [j for j in kept[i] if not rows[i]["correct"][j]] for i in ITEMS}
        pts.append({"k": k,
                    "problems_with_lesson": sum(bool(kept[i]) for i in ITEMS),
                    "problems_all_wrong": sum(bool(kept[i]) and len(wrong[i]) == len(kept[i]) for i in ITEMS),
                    "chains_kept": sum(len(kept[i]) for i in ITEMS),
                    "chains_wrong": sum(len(wrong[i]) for i in ITEMS)})
    curves[rule] = pts

for rule in curves:
    harness.table(
        f"rule: {rule}", ["attempts", "problems with a lesson", "all wrong", "chains kept", "chains wrong"],
        [[p["k"], p["problems_with_lesson"], p["problems_all_wrong"], p["chains_kept"], p["chains_wrong"]]
         for p in curves[rule] if p["k"] in (1, 2, 4, 8, 12, 16)])

GREEDY_LESSONS = sum(r["greedy_correct"] for r in rows.values())
print(f"\nSTaR's own setting, one greedy chain per problem: {GREEDY_LESSONS} problems yield a lesson")

## 6. The chains the vote missed, and the problems that never yield

`vote_missed`: a right chain was in the pile, and the most popular answer over all 16 was wrong or
tied. `never`: none of the 16 chains reached the answer key.

In [ ]:
vote_missed = [i for i in ITEMS if any(rows[i]["correct"])
               and not any(rows[i]["correct"][j] for j in harvest(rows[i], N_CHAINS, "popular"))]
never = [i for i in ITEMS if not any(rows[i]["correct"])]
print(f"{len(vote_missed)} problems where the key finds a lesson and the vote does not")
print(f"{len(never)} problems with no right chain in 16")

## 7. The figure

Left: problems that yield a lesson as attempts rise. Right: every problem at 16 attempts, under
each rule, ordered by how many of its chains were right.

In [ ]:
order = sorted(ITEMS, key=lambda i: (-sum(rows[i]["correct"]), i))
RIGHT, WRONG, NONE = 2, 1, 0


def grid(rule, k=N_CHAINS):
    cells = []
    for i in order:
        kept = harvest(rows[i], k, rule)
        cells.append(NONE if not kept else RIGHT if rows[i]["correct"][kept[0]] else WRONG)
    return [cells[r * 20:(r + 1) * 20] for r in range(len(cells) // 20)]


cmap = ListedColormap(["#ece8df", "#5b8fd1", "#6f8f55"])
fig = plt.figure(figsize=(12, 4.2))
ax = fig.add_axes([0.06, 0.15, 0.36, 0.75])
ks = [p["k"] for p in curves["key"]]
ax.plot(ks, [p["problems_with_lesson"] for p in curves["key"]], color="#6f8f55", lw=2, label="answer key")
ax.plot(ks, [p["problems_with_lesson"] - p["problems_all_wrong"] for p in curves["popular"]],
        color="#6f8f55", lw=1.5, ls="--", label="most popular answer, right")
ax.plot(ks, [p["problems_all_wrong"] for p in curves["popular"]], color="#5b8fd1", lw=1.5, ls="--",
        label="most popular answer, wrong")
ax.set_xlabel("attempts per problem")
ax.set_ylabel(f"problems with a lesson (of {len(ITEMS)})")
ax.set_xticks([1, 4, 8, 12, 16])
ax.set_ylim(0, len(ITEMS))
ax.legend(frameon=False, fontsize=8, loc="lower right")
for n, (rule, title) in enumerate([("key", "answer key"), ("popular", "most popular answer")]):
    g = fig.add_axes([0.5, 0.55 - n * 0.42, 0.46, 0.33])
    g.imshow(grid(rule), cmap=cmap, vmin=0, vmax=2)
    g.set_xticks([]), g.set_yticks([])
    g.set_title(f"{title}, {N_CHAINS} attempts", fontsize=9, loc="left")
fig.savefig(FIGURE_PATH, dpi=150)
plt.show()

## 8. Rationalization: the hint retry

STaR's second chance. Every problem is asked once more, greedily, with the right answer written
into the question as a hint. STaR only retries the problems it got wrong; asking all 100 lets the
page apply the retry to whichever problems a given number of attempts leaves without a right chain. A hinted chain that reaches the answer
becomes a lesson, recorded against the question without the hint. Grading is page 6's: a chain cut
off by its cap counts only if it already wrote a `\boxed{}` answer.

`mentions_hint` flags a chain that refers to being given the answer. With the hint deleted, such a
chain reads as an argument from nowhere.

In [ ]:
HINT_TEMPLATE = "{question}\n\n(Hint: the correct answer is {answer}.)"
MENTIONS_HINT = re.compile(r"\bhint\b|answer is given|given (?:that )?the (?:correct )?answer|we are told the answer",
                           re.IGNORECASE)

missed = [i for i in ITEMS if not rows[i]["greedy_correct"]]
HINT_ITEMS = missed[:QUICK_ITEMS] if RUN_MODE == "QUICK" else missed + [i for i in ITEMS if i not in missed]
HINT_RUN_ID = f"p12-hint-{SOURCE_RUN_ID}"
store = harness.TraceStore(HINT_PATH, HINT_RUN_ID, key_fields=("problem_id",))


def grade_chain(text, cut_off, reference):
    if cut_off:
        answer = (get_last_boxed(text) or "").strip().strip("$ ")
    else:
        answer = extract_final_candidate(text)
    return answer, bool(answer) and bool(grade_answer(answer, reference))


print(f"{len(missed)} problems the greedy chain missed" + (f"; {len(HINT_ITEMS)} to retry" if RUN_MODE != "REPLAY" else ""))
print(render_prompt(HINT_TEMPLATE.format(question="What is 2 + 2?", answer="4")))
store.report()

In [ ]:
if RUN_MODE != "REPLAY":
    import torch
    from reasoning_from_scratch.ch02 import get_device

    device = get_device()
    loaded = harness.load_model(MODEL_ID, device=device, dtype=harness.pick_dtype(device), warmup_tokens=16)
    model, tokenizer = loaded["model"], loaded["tokenizer"]
    harness.selftest(tokenizer)
    for i in harness.progress(HINT_ITEMS, desc="hint retry", unit="problem"):
        if store.get(i):
            continue
        r0 = pool[i][0]
        out = harness.generate(render_prompt(HINT_TEMPLATE.format(question=r0["question"], answer=r0["ground_truth"])),
                               model, tokenizer, enable_thinking=False, max_new_tokens=HINT_CAP)
        answer, ok = grade_chain(out["text"], out["truncated"], r0["ground_truth"])
        store.append({"problem_id": i, "model_id": MODEL_ID, "trace": out["text"],
                      "tokens_generated": out["tokens_generated"], "truncated": out["truncated"],
                      "answer": answer, "correct": ok, "mentions_hint": bool(MENTIONS_HINT.search(out["text"]))})
    del model
    harness.free_memory()
else:
    print("REPLAY: nothing generated")

In [ ]:
hint = {i: store.get(i) for i in ITEMS if store.get(i)}
if hint:
    recovered = [i for i in missed if i in hint and hint[i]["correct"]]
    HINT = {
        "run_id": HINT_RUN_ID,
        "template": HINT_TEMPLATE,
        "n_missed": len(missed),
        "n_retried": len(hint),
        "correct": {i: h["correct"] for i, h in hint.items()},
        "mentions": {i: h["mentions_hint"] for i, h in hint.items()},
        "recovered": recovered,
        "recovered_never": [i for i in recovered if i in never],
        "mentions_hint": [i for i in recovered if hint[i]["mentions_hint"]],
        "truncated": sum(h["truncated"] for h in hint.values()),
    }
    harness.table("The hint retry", ["", "problems"], [
        ["greedy chain missed", len(missed)],
        ["  of those, retried with the hint", sum(i in hint for i in missed)],
        ["  hinted chain reached the answer", len(recovered)],
        ["    of those, no right chain in page 7's 16", len(HINT["recovered_never"])],
        ["    of those, the chain mentions the hint", len(HINT["mentions_hint"])],
        ["all problems retried with the hint", len(hint)],
        ["  hinted chain reached the answer", sum(h["correct"] for h in hint.values())],
    ])
    print(f"\nlessons at one greedy attempt: {GREEDY_LESSONS} without the retry, "
          f"{GREEDY_LESSONS + len(recovered)} with it")
else:
    HINT = None
    print(f"no hinted chains on disk at {HINT_PATH}; run with PAGE12_RUN_MODE=HINT to make them")

## 9. Claims and results

The page may only say what these support.

In [ ]:
CLAIMS = {
    "key_keeps_no_wrong_chain": all(p["chains_wrong"] == 0 for p in curves["key"]),
    "popular_keeps_wrong_chains": curves["popular"][-1]["chains_wrong"] > 0,
    "more_attempts_more_lessons": curves["key"][-1]["problems_with_lesson"] > curves["key"][0]["problems_with_lesson"],
    "some_problems_never_yield": len(never) > 0,
    "vote_missed_what_key_found": len(vote_missed) > 0,
}
if HINT:
    CLAIMS["hint_recovers_lessons"] = len(HINT["recovered"]) > 0
    CLAIMS["hint_reaches_never"] = len(HINT["recovered_never"]) > 0
for k, v in CLAIMS.items():
    print(f"  {'yes' if v else 'NO ':<4}{k}")

RUN_ID = "p12-" + SOURCE_RUN_ID
results = {"run_id": RUN_ID, "source_run_id": SOURCE_RUN_ID, "pool_config": POOL_CONFIG,
           "greedy_config": GREEDY_CONFIG, "n_chains": N_CHAINS, "n_items": len(ITEMS),
           "greedy_lessons": GREEDY_LESSONS, "rows": rows, "curves": curves,
           "vote_missed": vote_missed, "never": never, "hint": HINT, "claims": CLAIMS}
harness.write_json(RESULTS_PATH, results)

r0 = pool[ITEMS[0]][0]
harness.write_manifest(
    MANIFEST_PATH, page=12, notebook="page_12_star_harvest.ipynb", run_id=RUN_ID, run_mode=RUN_MODE,
    source_run_id=SOURCE_RUN_ID, model_id=MODEL_ID, pool_config=POOL_CONFIG,
    temperature=r0["temperature"], chain_cap=r0["chain_budget"], n_items=len(ITEMS),
    n_chains=N_CHAINS, dataset="math500_l45",
    hint={"run_id": HINT_RUN_ID, "cap": HINT_CAP, "decoding": "greedy", "template": HINT_TEMPLATE,
          "n_retried": len(hint)})
print(f"\n{RESULTS_PATH}   {harness.kb(RESULTS_PATH):,.1f} KB")
print(f"{MANIFEST_PATH}")